# 05 - GGUF Export & Ollama Setup

Export fine-tuned model to GGUF format for local inference with Ollama.

**Prerequisites:** Run `03_training_qlora.ipynb` first.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
import os
os.chdir('/kaggle/working/fingpt-qlora')

In [ ]:
from unsloth import FastLanguageModel

EXPERIMENT_NAME = "v1_baseline"
lora_path = f"outputs/{EXPERIMENT_NAME}/lora_adapter"

# Load the base model + LoRA adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=lora_path,
    max_seq_length=2048,
    load_in_4bit=True,
)
print("Model loaded with LoRA adapter.")

## 1. Save as GGUF (q4_k_m)

In [ ]:
gguf_path = f"outputs/{EXPERIMENT_NAME}/gguf"

model.save_pretrained_gguf(
    gguf_path,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"\nGGUF model saved to: {gguf_path}")

# List files
import os
for f in os.listdir(gguf_path):
    size_mb = os.path.getsize(os.path.join(gguf_path, f)) / (1024*1024)
    print(f"  {f}: {size_mb:.1f} MB")

## 2. Create Ollama Modelfile

Copy the GGUF file to your local machine (WSL) and run:

```bash
# 1. Copy GGUF to local
# Download from Kaggle output or use kaggle CLI

# 2. Create Modelfile
cat > Modelfile << 'EOF'
FROM ./fingpt-qlora.Q4_K_M.gguf

TEMPLATE """{{ .System }}
{{ .Prompt }}"""

SYSTEM """You are FinGPT, a financial analysis assistant. You provide clear, structured analysis of financial data, reports, and market information."""

PARAMETER temperature 0.7
PARAMETER num_ctx 2048
PARAMETER top_p 0.9
EOF

# 3. Create Ollama model
ollama create fingpt -f Modelfile

# 4. Test
ollama run fingpt "Analyze the sentiment: Apple beats Q4 earnings estimates with record revenue"
```

In [ ]:
# Generate the Modelfile content
modelfile_content = '''FROM ./fingpt-qlora.Q4_K_M.gguf

TEMPLATE """{{ .System }}
{{ .Prompt }}"""

SYSTEM """You are FinGPT, a financial analysis assistant. You provide clear, structured analysis of financial data, reports, and market information."""

PARAMETER temperature 0.7
PARAMETER num_ctx 2048
PARAMETER top_p 0.9
'''

with open(f'{gguf_path}/Modelfile', 'w') as f:
    f.write(modelfile_content)

print(f"Modelfile saved to {gguf_path}/Modelfile")
print("\nTo use with Ollama:")
print(f"  1. Copy {gguf_path}/ to your local machine")
print(f"  2. Run: ollama create fingpt -f Modelfile")
print(f"  3. Run: ollama run fingpt")

## 3. Quick Inference Test

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Analyze the sentiment: Tesla shares surged 12% after record Q3 deliveries",
    "What is dollar-cost averaging and when should investors use it?",
    "Summarize: Revenue grew 15% YoY to $8.2B, EPS beat estimates by $0.30",
]

for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    
    print(f"\n{'='*60}")
    print(f"Q: {prompt}")
    print(f"{'='*60}")
    print(f"A: {response}")